## Feature Engineering

### Load Data

In [33]:
import numpy as np
import pandas as pd
import os
from pathlib import Path

# Read raw data
master_dir = os.path.dirname(os.getcwd())

cleaned_reviews = os.path.join(master_dir, "data", "cleaned_data", "cleaned_reviews.csv")

df_cleaned = pd.read_csv(cleaned_reviews)

In [34]:
# Add Review_ID column

df_cleaned.insert(
    0,
    "Review_ID",
    ["R{:06d}".format(i) for i in range(1, len(df_cleaned) + 1)]
)

df_cleaned.head(10)

,Review_ID,Review,Rating,Lemmatized_Tokens,Review_lemmatized,Platform,Restaurant,Sentiment_Label
0,R000001,food is taste good and the environment also no...,3.0,"['food', 'taste', 'good', 'environment', 'also...",food taste good environment also not bad . how...,Google,Tandoor Grill,Neutral
1,R000002,our family celebrated birthday on last weekend...,1.0,"['family', 'celebrate', 'birthday', 'last', 'w...",family celebrate birthday last weekend think 5...,Tripadvisor,Latest Recipe,Negative
2,R000003,pretty nice nonya food but very commercial alr...,3.0,"['pretty', 'nice', 'nonya', 'food', 'but', 've...",pretty nice nonya food but very commercial alr...,Google,Madam Kwan's Suria KLCC,Neutral
3,R000004,"food was not on point, the seafood was frozen,...",2.0,"['food', 'not', 'point', ',', 'seafood', 'froz...","food not point , seafood frozen , seasoning , ...",Google,Mercat Barcelona Gastrobar (1MontKiara),Negative
4,R000005,personaly for me is a big no....i saw flies la...,1.0,"['personaly', 'big', 'no', '....', 'saw', 'fly...",personaly big no .... saw fly lay egg fry chic...,Google,Restoran DEEN | Nasi Kandar,Negative
5,R000006,"drinks are slow, service is below average, sta...",1.0,"['drink', 'slow', ',', 'service', 'average', '...","drink slow , service average , staff not exact...",Google,"WET® Deck, W Kuala Lumpur",Negative
6,R000007,"service was good, food could be better. din ta...",3.0,"['service', 'good', ',', 'food', 'could', 'bet...","service good , food could better . din tai fun...",Google,DIN by Din Tai Fung,Neutral
7,R000008,-food bento_box very good.-privacy every table...,5.0,"['-food', 'bento_box', 'very', 'good.-privacy'...",-food bento_box very good.-privacy every table...,Google,Ishin Japanese Dining,Positive
8,R000009,the chicken macaroni pie is to die for! the ch...,4.0,"['chicken', 'macaroni', 'pie', 'die', '!', 'ch...",chicken macaroni pie die ! chicken stew very f...,Tripadvisor,Yeng Keng Cafe,Positive
9,R000010,"it is a well known restaurant in ipoh, the pub...",1.0,"['well', 'know', 'restaurant', 'ipoh', ',', 'p...","well know restaurant ipoh , public trust belie...",Google,Anderson Curry House,Negative


### TF-IDF Vectorization

In [35]:
# check is that any missing value in the lemmatized review column
print(df_cleaned["Review_lemmatized"].isna().sum())

6


In [36]:
df_cleaned[df_cleaned["Review_lemmatized"].isna()][
    ["Review", "Review_lemmatized"]
].head()

,Review,Review_lemmatized
2145,y you,NaN
20998,i,NaN
46309,up,NaN
61835,just do it,NaN
81075,out,NaN


In [37]:
# remove the rows with missing values in the lemmatized review column
df_cleaned = df_cleaned.dropna(subset=["Review_lemmatized"])

In [38]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.9
)

tfidf_matrix  = tfidf_vectorizer.fit_transform(df_cleaned["Review_lemmatized"])

In [39]:
# check the shape of the TF-IDF matrix
print("TF-IDF shape:", tfidf_matrix.shape)

TF-IDF shape: (96606, 5000)


In [40]:
feature_names = tfidf_vectorizer.get_feature_names_out()

print("Total vocabulary size:", len(feature_names))

Total vocabulary size: 5000


In [41]:
mean_tfidf = np.asarray(tfidf_matrix.mean(axis=0)).flatten()

top_indices = mean_tfidf.argsort()[::-1][:20]

top_tfidf_features = pd.DataFrame({
    "Feature": feature_names[top_indices],
    "Mean_TFIDF": mean_tfidf[top_indices]
})

top_tfidf_features

,Feature,Mean_TFIDF
0,food,0.045162
1,not,0.041302
2,good,0.037459
3,but,0.028145
4,service,0.024798
5,very,0.023980
6,nice,0.023795
7,place,0.022874
8,so,0.020789
9,price,0.018365


### Unigrams & Bigrams

In [42]:
unigrams = [
    feature for feature in feature_names
    if len(feature.split()) == 1
]

bigrams = [
    feature for feature in feature_names
    if len(feature.split()) == 2
]

print("Number of unigrams:", len(unigrams))
print("Number of bigrams:", len(bigrams))

Number of unigrams: 2704
Number of bigrams: 2296


In [43]:
# get the top unigrams
unigram_indices = [
    i for i, feature in enumerate(feature_names)
    if len(feature.split()) == 1
]

top_unigram_indices = sorted(
    unigram_indices,
    key=lambda i: mean_tfidf[i],
    reverse=True
)[:20]

top_unigrams = pd.DataFrame({
    "Feature": feature_names[top_unigram_indices],
    "Mean_TFIDF": mean_tfidf[top_unigram_indices]
})

top_unigrams

,Feature,Mean_TFIDF
0,food,0.045162
1,not,0.041302
2,good,0.037459
3,but,0.028145
4,service,0.024798
5,very,0.023980
6,nice,0.023795
7,place,0.022874
8,so,0.020789
9,price,0.018365


In [44]:
# get the top bigrams
bigram_indices = [
    i for i, feature in enumerate(feature_names)
    if len(feature.split()) == 2
]

top_bigram_indices = sorted(
    bigram_indices,
    key=lambda i: mean_tfidf[i],
    reverse=True
)[:20]

top_bigrams = pd.DataFrame({
    "Feature": feature_names[top_bigram_indices],
    "Mean_TFIDF": mean_tfidf[top_bigram_indices]
})

top_bigrams

,Feature,Mean_TFIDF
0,show less,0.013542
1,good food,0.008636
2,food good,0.006540
3,food not,0.006150
4,but not,0.005558
5,nice food,0.005365
6,so so,0.005213
7,good service,0.005206
8,very good,0.005101
9,good but,0.004740


### Export Feature Data

In [45]:
# Project directory
project_dir = os.path.dirname(os.getcwd())

feature_info = pd.DataFrame({
    "Feature_Index": range(len(feature_names)),
    "Feature": feature_names,
    "Mean_TFIDF": mean_tfidf
})


feature_info.to_csv(
    os.path.join(
        project_dir,
        "data",
        "processed_data",
        "tfidf_features.csv"
    ),
    index=False
)

### Save TF-IDF Metric

In [46]:
from scipy.sparse import save_npz

tfidf_path = os.path.join(
    project_dir,
    "data",
    "processed_data",
    "tfidf_matrix.npz"
)

save_npz(tfidf_path, tfidf_matrix)

print("TF-IDF matrix saved to:", tfidf_path)

TF-IDF matrix saved to: c:\Users\user\Downloads\social_computing_assignment\data\processed_data\tfidf_matrix.npz


### Save Feature Engineered Reviews

In [47]:
feature_data_path = os.path.join(
    project_dir,
    "data",
    "processed_data",
    "feature_engineered_reviews.csv"
)

df_cleaned.to_csv(
    feature_data_path,
    index=False
)

print("Feature-engineered dataset saved to:", feature_data_path)

Feature-engineered dataset saved to: c:\Users\user\Downloads\social_computing_assignment\data\processed_data\feature_engineered_reviews.csv


### Save TF-IDF Vectorizer

In [48]:
import joblib

vectorizer_path = os.path.join(
    project_dir,
    "data",
    "processed_data",
    "tfidf_vectorizer.pkl"
)

joblib.dump(
    tfidf_vectorizer,
    vectorizer_path
)

print("TF-IDF vectorizer saved to:", vectorizer_path)

TF-IDF vectorizer saved to: c:\Users\user\Downloads\social_computing_assignment\data\processed_data\tfidf_vectorizer.pkl
